# Imbalanced Breast Cancer Analysis: SVM & Decision Trees

## 1. Problem Statement
**Goal**: Implement and evaluate SVM and Decision Trees on a highly imbalanced (90:10) Breast Cancer dataset.

**Steps to Solve**:
1.  **Data Preparation**: Load dataset, recreate imbalance by undersampling Malignant class.
2.  **Model Training**: Train Baseline, Class-Weighted (Balanced), and Custom-Weighted versions of SVM and DT.
3.  **Evaluation**: Compare Precision, Recall, F1, and ROC-AUC.

**Expected Output**:
*   Visual confusion matrices for all 6 models.
*   ROC Curves overlay.
*   Final performance table highlighting Recall improvement with weighting.


## 1. Imports
### Code Explanation
*   **2.1 Definition**: Importing NumPy.
*   **2.2 Why it is used**: For array manipulation and random sampling.
*   **2.3 When to use**: Essential for scientific computing.
*   **2.4 Where to use**: Global scope.
*   **2.5 How to use**: `import numpy as np`.
*   **2.6 How it works**: Loads the C-optimized math library.
*   **2.7 Output**: Module `np`.


In [1]:
import numpy as np


### Code Explanation
*   **2.1 Definition**: Importing Pandas.
*   **2.2 Why it is used**: For creating the final results table.
*   **2.7 Output**: Module `pd`.


In [2]:
import pandas as pd


### Code Explanation
*   **2.1 Definition**: Importing Matplotlib Pyplot.
*   **2.2 Why it is used**: For plotting ROC curves.
*   **2.7 Output**: Module `plt`.


In [3]:
import matplotlib.pyplot as plt


### Code Explanation
*   **2.1 Definition**: Importing Seaborn.
*   **2.2 Why it is used**: For plotting Heatmaps (Confusion Matrices).
*   **2.7 Output**: Module `sns`.


In [4]:
import seaborn as sns


### Code Explanation
*   **2.1 Definition**: Importing Dataset Loader.
*   **2.2 Why it is used**: To get the Breast Cancer data.
*   **2.7 Output**: Function `load_breast_cancer`.


In [5]:
from sklearn.datasets import load_breast_cancer


### Code Explanation
*   **2.1 Definition**: Importing Classifiers.
*   **2.2 Why it is used**: We need SVC and DecisionTree for the comparison.
*   **2.7 Output**: Classes `SVC`, `DecisionTreeClassifier`.


In [6]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


### Code Explanation
*   **2.1 Definition**: Importing Splitter.
*   **2.2 Why it is used**: To split Train/Test sets.
*   **2.7 Output**: Function `train_test_split`.


In [ ]:
from sklearn.model_selection import train_test_split


### Code Explanation
*   **2.1 Definition**: Importing Metrics.
*   **2.2 Why it is used**: To calculate performance.
*   **2.7 Output**: Functions like `confusion_matrix`, `roc_auc_score`.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, f1_score


## 2. Dataset Creation
### Code Explanation
*   **2.1 Definition**: Loading the raw data.
*   **2.7 Output**: Bunch object containing data.


In [ ]:
data = load_breast_cancer()


### Code Explanation
*   **2.1 Definition**: Extracting Features (X).
*   **2.7 Output**: Array of shape (569, 30).


In [ ]:
X_raw = data.data


### Code Explanation
*   **2.1 Definition**: Extracting Target (y).
*   **2.7 Output**: Array of shape (569,).


In [ ]:
y_raw = data.target


## 3. Creating Imbalance (90:10)
### Code Explanation
*   **2.1 Definition**: Finding indices of Class 0 (Malignant).
*   **2.6 How it works**: `np.where(condition)` returns indices where condition is true.
*   **2.7 Output**: Array of indices.


In [ ]:
idx_0 = np.where(y_raw == 0)[0]


### Code Explanation
*   **2.1 Definition**: Finding indices of Class 1 (Benign).
*   **2.7 Output**: Array of indices.


In [ ]:
idx_1 = np.where(y_raw == 1)[0]


### Code Explanation
*   **2.1 Definition**: Calculating count of Benign samples.
*   **2.7 Output**: Integer (357).


In [ ]:
n_benign = len(idx_1)


### Code Explanation
*   **2.1 Definition**: Calculating target count for Malignant samples to simulate 10% ratio.
*   **2.6 How it works**: `n_benign / 0.9 * 0.1`.
*   **2.7 Output**: Integer (39).


In [ ]:
n_malignant_target = int(n_benign / 0.9 * 0.1)


### Code Explanation
*   **2.1 Definition**: Setting Random Seed.
*   **2.2 Why it is used**: Reproducibility.
*   **2.7 Output**: None.


In [ ]:
np.random.seed(42)


### Code Explanation
*   **2.1 Definition**: Randomly selecting 39 Malignant indices.
*   **2.6 How it works**: Random sampling without replacement.
*   **2.7 Output**: Array of 39 indices.

### Argument Explanation: `replace`
*   **3.1 Definition**: Whether to sample with replacement.
*   **3.6 Output**: `replace=False` ensures unique patients.


In [ ]:
idx_0_undersampled = np.random.choice(idx_0, size=n_malignant_target, replace=False)


### Code Explanation
*   **2.1 Definition**: Merging the index arrays.
*   **2.7 Output**: Combined array of 396 indices.


In [ ]:
idx_final = np.concatenate([idx_1, idx_0_undersampled])


### Code Explanation
*   **2.1 Definition**: Selecting final X.
*   **2.7 Output**: (396, 30).


In [ ]:
X = X_raw[idx_final]


### Code Explanation
*   **2.1 Definition**: Selecting final y.
*   **2.7 Output**: (396,).


In [ ]:
y = y_raw[idx_final]


### Code Explanation
*   **2.1 Definition**: Splitting Data.
*   **3.6 Output**: 4 arrays.

### Argument Explanation: `stratify`
*   **3.1 Definition**: Maintain class ratios.
*   **3.2 Why it is used**: Critical for imbalanced data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


### Code Explanation
*   **2.1 Definition**: Checking final class distribution.
*   **2.7 Output**: Counts [7, 72] in test set.


In [ ]:
print(f'Test Counts: {np.bincount(y_test)}')


## 4. Training Loop
### Code Explanation
*   **2.1 Definition**: Defining Dictionary of Models.
*   **2.7 Output**: Dictionary.


In [ ]:
models = {
    'SVM_Baseline': SVC(kernel='linear', probability=True, random_state=42),
    'SVM_Balanced': SVC(kernel='linear', class_weight='balanced', probability=True, random_state=42),
    'SVM_Custom':   SVC(kernel='linear', class_weight={0: 9, 1: 1}, probability=True, random_state=42),
    'DT_Baseline':  DecisionTreeClassifier(random_state=42),
    'DT_Balanced':  DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'DT_Custom':    DecisionTreeClassifier(class_weight={0: 9, 1: 1}, random_state=42)
}


### Code Explanation
*   **2.1 Definition**: Initializing results list.
*   **2.7 Output**: Empty list.


In [ ]:
results = []


### Code Explanation
*   **2.1 Definition**: Initializing plotting figure.
*   **2.7 Output**: Figure object.


In [ ]:
plt.figure(figsize=(18, 10))


### Code Explanation
*   **2.1 Definition**: Iterating through models.
*   **2.6 How it works**: Fits, Predicts, Calculates Metrics, and Plots.
*   **2.7 Output**: Populated results list.


In [ ]:
for i, (name, model) in enumerate(models.items()):
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 0]
    
    # Metrics
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    recall = cm[0,0] / (cm[0,0] + cm[0,1])
    precision = cm[0,0] / (cm[0,0] + cm[1,0]) if (cm[0,0] + cm[1,0]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    roc_auc = roc_auc_score(y_test == 0, y_prob)
    
    results.append({'Model': name, 'Recall': recall, 'F1': f1, 'AUC': roc_auc})
    
    # Plot
    plt.subplot(2, 3, i+1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Mal', 'Ben'], yticklabels=['Mal', 'Ben'])
    plt.title(f'{name} | Recall: {recall:.2f}')


In [ ]:
plt.tight_layout()
plt.show()


## 5. Final Results
### Code Explanation
*   **2.1 Definition**: Displaying results table.
*   **2.7 Output**: DataFrame.


In [ ]:
pd.DataFrame(results)
